# 🐍 01 · Python & NumPy for biologists
### Your first programs: DNA strings and the genetic code, genotype matrices, the leukaemia table in pandas, plots, error messages — and a Hardy–Weinberg mini-project

**BiologyToML** · Level 1 · 02-620 core · Part A — Foundations: from biology to data · Module 01 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/01_python_numpy_for_biologists.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib |
| **Prerequisites** | Module 00 (running cells in Jupyter/Colab). No programming experience needed. |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | The programming prerequisite of 02-620 (CMU 02-601 *Programming for Scientists*): Python and NumPy at the level the lectures assume — and why the course uses Python; the N × J data matrix with genotypes coded 0/1/2 (the regression lecture's setting); expression profiles as inputs X and diagnoses as outputs Y (the leukaemia case study of the introductory lecture) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The leukaemia expression table in Sections 5–6 is **SIMULATED** with the toolkit's
>     `simulate_leukemia_expression` (72 patients × 3,571 genes shaped like Golub *et al.* 1999; the values are not the
>     paper's). All genotype matrices are **simulated** under Hardy–Weinberg equilibrium with the course toolkit. The
>     only real biological sequence is a short public one — the 333-nucleotide coding sequence of human preproinsulin
>     and its 110-residue protein (UniProt P01308) — typed into a cell of Section 2 as a worked example; nothing is
>     downloaded.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Write** Python with numbers, strings, booleans, lists, dictionaries, `for` loops, `if` statements and functions with docstrings.
2. **Implement** classic sequence tools — nucleotide counts, GC content, reverse complement, translation with the genetic code — and **use** them to study point mutations in the insulin gene.
3. **Create and manipulate** NumPy arrays: shape, dtype, indexing, slicing, boolean masks, `axis=` reductions and broadcasting, using an N × J genotype matrix.
4. **Explain** why vectorised NumPy code beats Python loops, and **measure** the speed-up.
5. **Generate** reproducible random numbers with a seeded generator and **simulate** coin flips and diploid genotypes.
6. **Explore** a gene-expression table with pandas (`head`, `describe`, selection, `groupby`) and **find** the genes that differ most between ALL and AML.
7. **Plot** histograms, scatter plots, heatmaps and line plots with labelled axes, and **read** error messages to debug code.
8. **Compute** allele frequencies and Hardy–Weinberg expected heterozygosity for 100 SNPs, and **interpret** a heterozygote deficit.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports and course helpers | 5 min |
| 1 · Values and variables | numbers, a DNA string, booleans, f-strings | 12 min |
| 2 · Lists, dictionaries, loops, functions | GC content, reverse complement, the genetic code, insulin | 25 min |
| 3 · NumPy arrays | a 200 × 1,000 genotype matrix: shape, indexing, masks, axis, broadcasting, speed | 22 min |
| 4 · Random numbers and reproducibility | seeds, coin flips, genotypes from two allele draws | 10 min |
| 5 · pandas | the leukaemia table: select, summarise, group, rank genes; 🧪 true effects | 15 min |
| 6 · Matplotlib | histogram, scatter, heatmap, line plot | 10 min |
| 7 · Reading error messages | NameError, IndexError, KeyError, TypeError, shape mismatch, silent overflow | 8 min |
| 8 · Mini-project | Hardy–Weinberg across 100 SNPs; a hidden population structure | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 3 min |

## 🧫 The biological question

A single run of a modern sequencer produces billions of DNA letters. A genotyping chip reports hundreds of thousands of
SNPs for each person; an expression experiment, thousands of genes for each sample. A biologist who can write a few
lines of code can ask questions of such data in minutes that would take weeks by hand — *what fraction of this genome
is G or C? which genes differ between two leukaemias? are genotype frequencies what Mendel predicts?*

02-620 assumes exactly this skill. Its first lecture lists the prerequisites: *basic familiarity with Python and NumPy
programming*, *simple linear algebra using NumPy*, and *saving and accessing Jupyter notebooks* — the content of CMU's
02-601 *Programming for Scientists*. Every 02-620 homework must be written in Python.

This module teaches precisely that Python, always with biological data: the DNA of the insulin gene, a matrix of
genotypes, and the leukaemia expression table (simulated, shaped like Golub's) you met in Module 00. You will finish with a small population-genetics
analysis — checking Hardy–Weinberg proportions across 100 SNPs — and discover how a hidden population structure leaves
a fingerprint in the data. Type, run, change, break, fix: that is how programming is learned.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_genotypes` creates genotype matrices under Hardy–Weinberg equilibrium,
               `simulate_leukemia_expression` generates the course's leukaemia-like expression table (with its
               hidden truth), `standardize` converts columns to z-scores and `CLASS_COLORS` fixes the course's
               colours. Once you finish this module you will be
               able to read every line of this cell. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np

LEUKEMIA_MARKERS = {
    "CST3": ("AML", "cystatin C"), "ZYX": ("AML", "zyxin"), "CD33": ("AML", "CD33 antigen"),
    "MPO": ("AML", "myeloperoxidase"), "LYZ": ("AML", "lysozyme"), "CTSD": ("AML", "cathepsin D"),
    "FAH": ("AML", "fumarylacetoacetate hydrolase"), "LTC4S": ("AML", "leukotriene C4 synthase"),
    "LYN": ("AML", "LYN kinase"), "HOXA9": ("AML", "homeobox A9"), "CFD": ("AML", "adipsin"),
    "LEPR": ("AML", "leptin receptor"), "SRGN": ("AML", "proteoglycan 1 (serglycin)"),
    "CXCL8": ("AML", "interleukin-8"), "AZU1": ("AML", "azurocidin"), "MCL1": ("AML", "MCL1"),
    "ITGAX": ("AML", "CD11c"), "NFKBIA": ("AML", "MAD-3 (IκBα)"), "ELANE": ("AML", "neutrophil elastase"),
    "CEBPA": ("AML", "C/EBPα"), "CSF3R": ("AML", "G-CSF receptor"), "FCER1G": ("AML", "Fc receptor γ"),
    "TYROBP": ("AML", "DAP12"), "S100A9": ("AML", "calprotectin subunit"), "CFP": ("AML", "properdin"),
    "MYB": ("ALL", "c-Myb"), "CCND3": ("ALL", "cyclin D3"), "RBBP4": ("ALL", "RbAp48"),
    "SMARCA4": ("ALL", "SNF2 (BRG1)"), "TCF3": ("ALL", "E2A"), "DUT": ("ALL", "dUTPase"),
    "STMN1": ("ALL", "Op18 (stathmin)"), "IL7R": ("ALL", "IL-7 receptor"), "RABEP1": ("ALL", "rabaptin-5"),
    "CBX3": ("ALL", "heterochromatin protein p25"), "DNTT": ("ALL", "terminal deoxynucleotidyl transferase"),
    "CD24": ("ALL", "CD24"), "TOP2B": ("ALL", "topoisomerase II β"), "IRF2": ("ALL", "IRF-2"),
    "MYL6": ("ALL", "myosin light chain"), "PSMB8": ("ALL", "proteasome subunit"), "LMO2": ("ALL", "LMO2"),
    "CD79A": ("B-ALL", "MB-1 (Igα)"), "CD19": ("B-ALL", "CD19"), "CD79B": ("B-ALL", "Igβ"),
    "VPREB1": ("B-ALL", "pre-B receptor"), "PAX5": ("B-ALL", "PAX5"), "CD22": ("B-ALL", "CD22"),
    "CD3E": ("T-ALL", "CD3ε"), "CD3D": ("T-ALL", "CD3δ"), "CD2": ("T-ALL", "CD2"), "CD7": ("T-ALL", "CD7"),
    "LCK": ("T-ALL", "LCK kinase"), "CD1E": ("T-ALL", "CD1e"),
}


def simulate_leukemia_expression(seed: int = 1999) -> dict:
    """Synthetic leukaemia expression data shaped like the classic Golub et al. (1999) study.

    72 bone-marrow/blood samples — 47 acute lymphoblastic leukaemia (ALL: 38 B-lineage, 9 T-lineage)
    and 25 acute myeloid leukaemia (AML) — × 3,571 genes on a log10 microarray scale, with the
    original 38/34 training/test design (train: 19 B-ALL, 8 T-ALL, 11 AML).  Values below the
    detection limit are floored at 2.0 (≈ 23 % of entries) and saturate at 4.2.  The 54 genes of
    ``LEUKEMIA_MARKERS`` carry the strongest signals (42 differ between ALL and AML by 0.5–1.0
    log10 units; 12 mark B- or T-lineage ALL), 900 more genes differ modestly between ALL and AML,
    150 separate T-ALL from B-ALL, and 15 hidden co-expression programmes correlate genes.  Two
    AML samples are atypical (signature at 35 % strength).  The classes are cleaner than in the
    real study, so many classifiers make few or no errors.

    Returns a dict with ``X`` (72 × 3,571), ``y`` (0 = ALL, 1 = AML), ``labels``, ``subtype``,
    ``genes``, ``is_train``, ``sample_id`` (1–72), ``atypical`` (True for the two weak-signature AML
    samples), ``true_effect`` and ``source``.  ``true_effect`` is the planted AML − ALL difference
    per gene; it is 0 for the 12 lineage markers and the 150 T-vs-B genes, although the B-lineage
    markers still differ between ALL and AML through the subtype shifts (by about 0.8 log10 units).
    The default ``seed`` gives the course's standard data set.
    """
    rng = np.random.default_rng(seed)
    J = 3571
    subtype = np.array(["B-ALL"] * 19 + ["T-ALL"] * 8 + ["AML"] * 11 + ["B-ALL"] * 19 + ["T-ALL"] * 1 + ["AML"] * 14,
                       dtype=object)
    is_train = np.r_[np.ones(38, bool), np.zeros(34, bool)]
    order = np.r_[rng.permutation(38), 38 + rng.permutation(34)]          # mix classes within each set
    subtype, is_train = subtype[order], is_train[order]
    n = len(subtype)
    y = (subtype == "AML").astype(int)
    named = list(LEUKEMIA_MARKERS)
    genes = np.array(named + [f"G{j:04d}" for j in range(1, J - len(named) + 1)], dtype=object)
    low = rng.random(J) < 0.4                                             # genes mostly below detection
    mu = np.where(low, rng.uniform(1.5, 2.4, J), np.clip(rng.normal(2.9, 0.4, J), 2.2, 4.0))
    sd = rng.uniform(0.12, 0.35, J)
    X = mu + sd * rng.normal(0, 1, (n, J))
    factors = rng.normal(0, 1, (n, 15))                                  # co-expression programmes
    loadings = rng.normal(0, 0.15, (15, J)) * (rng.random((15, J)) < 0.08)
    X += factors @ loadings
    effect = np.zeros(J)
    strong = np.arange(len(named))
    for j, g in enumerate(named):
        cls = LEUKEMIA_MARKERS[g][0]
        size = rng.uniform(0.5, 1.0)
        effect[j] = size if cls == "AML" else -size if cls == "ALL" else 0.0
        mu[j] = max(mu[j], 2.5)
    moderate = rng.choice(np.arange(len(named), J), 900, replace=False)
    effect[moderate] = rng.choice([-1, 1], 900) * rng.uniform(0.1, 0.4, 900)
    strength = np.ones(n)
    strength[np.flatnonzero(y == 1)[rng.choice((y == 1).sum(), 2, replace=False)]] = 0.35   # atypical AML
    X[:, strong] = mu[strong] + sd[strong] * rng.normal(0, 1, (n, len(strong)))
    X += np.outer((y - y.mean()) * strength, effect)
    for g, (cls, _) in LEUKEMIA_MARKERS.items():
        if cls in ("B-ALL", "T-ALL"):
            j = named.index(g)
            X[:, j] += np.where(subtype == cls, 0.9, np.where(subtype == "AML", -0.2, -0.5))
    lineage = rng.choice(np.setdiff1d(np.arange(len(named), J), moderate), 150, replace=False)
    X[:, lineage] += np.outer(np.where(subtype == "T-ALL", 1.0, np.where(subtype == "B-ALL", -0.2, 0.0)),
                              rng.choice([-1, 1], 150) * rng.uniform(0.4, 0.8, 150))
    X = np.round(np.clip(X, 2.0, np.log10(16000)), 4)
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL").astype(object), subtype=subtype, genes=genes,
                is_train=is_train, sample_id=np.arange(1, n + 1), atypical=strength < 1, true_effect=effect,
                source="SIMULATED leukaemia data (shaped like Golub et al. 1999)")


def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


print("🧰 Toolkit loaded:", ", ".join(['simulate_leukemia_expression', 'simulate_genotypes', 'standardize']))

---
## 1 · Values and variables

> 📍 **Where we are.** In Module 00 you *ran* code. Now you start *writing* it. Everything in Python is built from a
> few kinds of values — numbers, text, true/false — stored under names.

A **variable** is a name attached to a value: `n_cells = 1` puts the value 1 in a box labelled `n_cells`. The `=` sign
means *assign*, not *equals* — `n_cells = n_cells * 2` is perfectly sensible: "take the old value, double it, store it
back". Names may contain letters, digits and underscores, must not start with a digit, and are case-sensitive
(`DNA` and `dna` are different variables).

**Numbers** come in two kinds: **integers** (`int`: 0, 42, −3) and **floating-point numbers** (`float`: 0.5, 3.0,
6.02e23 — the `e23` means "× 10²³"). The operators are `+ - * /`, `**` for powers, `//` for division rounded down and
`%` for the remainder.

🧮 **Math refresher — powers and remainders.** $2^{10}$ ("2 to the power 10") means 2 multiplied by itself 10 times,
$2 \times 2 \times \dots \times 2 = 1024$; in Python, `2 ** 10`. The **remainder** of 17 divided by 3 is 2, because
$17 = 5 \times 3 + 2$; in Python, `17 % 3` gives 2 and `17 // 3` gives 5. A number is divisible by 3 exactly when its
remainder is 0 — useful, because a coding sequence has a length divisible by 3 (one codon = 3 bases).

In [ ]:
# ── 1.1 Numbers: bacterial growth ──
doubling_time_min = 20                      # E. coli in rich medium divides about every 20 minutes
hours = 8
divisions = hours * 60 // doubling_time_min # whole number of divisions in 8 hours
cells_after = 2 ** divisions                # one cell → 2 → 4 → 8 → ...
print("divisions:", divisions)
print("cells after", hours, "hours:", cells_after)
print("type of divisions:", type(divisions), "| type of 20 / 3:", type(20 / 3), "=", 20 / 3)
print("17 // 3 =", 17 // 3, "and 17 % 3 =", 17 % 3)

🔍 **What just happened?** 8 hours × 60 minutes ÷ 20 minutes = **24 divisions**, so one bacterium could become
$2^{24}$ = **16,777,216** cells (in reality nutrients run out first). `//` and `**` with integers give integers, while
`/` always gives a float, even for 20 / 3 = 6.666…. `type(...)` tells you what kind of value something is.

### 1.2 Text: a DNA sequence as a string

Text values are **strings** (`str`), written between quotes. A DNA sequence is naturally a string. Each character has a
position called its **index** — and Python counts from **0**, not 1. `seq[0]` is the first base, `seq[-1]` the last.
A **slice** `seq[a:b]` takes positions a, a+1, …, b−1 (the end is *excluded*), so `seq[0:3]` is the first codon.

In [ ]:
# ── 1.2 Strings: the first codons of the insulin gene ──
dna = "ATGGCCCTGTGGATGCGCCTCCTGCCC"            # the first 27 bases of a coding sequence for human insulin
print("sequence:", dna)
print("length:", len(dna), "bases")
print("first base dna[0]:", dna[0], "| last base dna[-1]:", dna[-1])
print("first codon dna[0:3]:", dna[0:3], "| second codon dna[3:6]:", dna[3:6])
print("number of G:", dna.count("G"), "| lower case:", dna.lower()[:9], "...")
print("glued together:", dna[0:3] + "..." + dna[-3:])

🔍 **What just happened?** The string has 27 characters. `dna[0]` is `A` and `dna[0:3]` is `ATG` — the **start codon**
that begins almost every protein-coding sequence. Strings come with built-in **methods**, called with a dot:
`dna.count("G")` counts Gs, `dna.lower()` makes a lower-case copy. `+` glues strings together.

⚠️ **Off by one.** Biologists number bases from 1; Python from 0. Base number 4 of the sequence (biologist's counting)
is `dna[3]` in Python. Slices exclude their end: `dna[3:6]` is bases 4–6. Almost everyone makes this mistake at first.

### 1.3 True or false: booleans and comparisons

A comparison produces a **boolean** (`bool`): either `True` or `False`. Comparisons are `==` (equal — two signs!),
`!=` (not equal), `<`, `<=`, `>`, `>=`; the word `in` asks whether something occurs inside a string or list. Booleans
can be combined with `and`, `or`, `not`.

In [ ]:
# ── 1.3 Booleans: is this a plausible open reading frame? ──
starts_with_atg = dna.startswith("ATG")
length_ok = len(dna) % 3 == 0                 # a whole number of codons?
has_taa = "TAA" in dna
print("starts with ATG:", starts_with_atg)
print("length divisible by 3:", length_ok)
print("contains the letters TAA somewhere:", has_taa)
print("plausible start of an ORF:", starts_with_atg and length_ok)
print("True counts as 1 when added:", True + True + False)

🔍 **What just happened?** Each line produced `True` or `False`. The fragment starts with ATG and its length (27) is
divisible by 3. In arithmetic `True` behaves like 1 and `False` like 0 — so adding booleans *counts* how many are
true. You will use this trick constantly with NumPy (Section 3).

### 1.4 Printing nicely: f-strings

An **f-string** (a string with an `f` in front) replaces each `{expression}` by its value; after a colon you can
specify a format: `{x:.2f}` = 2 decimals, `{x:.1%}` = percentage with 1 decimal, `{n:,}` = thousands separators,
`{s:>8}` = right-aligned in 8 characters.

In [ ]:
# ── 1.4 f-strings ──
gc = (dna.count("G") + dna.count("C")) / len(dna)
print(f"GC content of the fragment: {gc:.3f}  =  {gc:.1%}")
print(f"cells after {hours} h: {cells_after:,}")
print(f"{'gene':>8} | {'length':>6}")
print(f"{'INS':>8} | {len(dna):>6}")

🔍 **What just happened?** The **GC content** — the fraction of bases that are G or C — of this fragment is
0.704 = 70.4 %. GC content matters biologically: G–C pairs form three hydrogen bonds (A–T pairs two), so GC-rich DNA
melts at higher temperature, and GC content differs systematically between genomes and between regions of a genome.

✏️ **Your turn.** Change `doubling_time_min` in cell 1.1 to 30 (slower growth) and re-run cells 1.1 and 1.4. How many
cells now? Then compute, in a new cell (*Insert → Code cell*, or the **+ Code** button), how many divisions are needed
to reach one billion ($10^9$) cells. Hint: try `2 ** 29` and `2 ** 30`.

---
## 2 · Lists, dictionaries, loops, conditions and functions

> 📍 **Where we are.** Section 1 gave you single values. Real analyses repeat the same step many times (for every base,
> every codon, every gene) and look things up in tables (the genetic code). Five tools make that possible.

### 2.1 Lists: ordered collections

A **list** holds several values in order, between square brackets: `bases = ["A", "C", "G", "T"]`. Lists are indexed
and sliced exactly like strings, but unlike strings they can be **changed**: `.append(x)` adds an element at the end.

In [ ]:
# ── 2.1 Lists ──
bases = ["A", "C", "G", "T"]
stop_codons = ["TAA", "TAG"]
stop_codons.append("TGA")                         # lists can grow
print("bases:", bases, "| number of bases:", len(bases))
print("first and last:", bases[0], bases[-1], "| the purines A and G are bases[0] and bases[2]:", bases[0], bases[2])
print("stop codons:", stop_codons)
print("is 'TGA' a stop codon?", "TGA" in stop_codons)

🔍 **What just happened?** Two lists: the four bases and the three **stop codons** (TAA, TAG, TGA — signals that end
translation). The third stop codon was added with `.append`. `in` works on lists too.

### 2.2 Loops: do something for every element

A `for` loop repeats an indented block once for each element of a sequence:

```python
for base in dna:        # base takes the values 'A', 'T', 'G', ... one after another
    ...                 # this indented block runs once per base
```

**Indentation is part of the language**: the lines that belong to the loop are indented by 4 spaces. `range(n)` gives
the numbers 0, 1, …, n−1, and `range(start, stop, step)` counts in steps — `range(0, 27, 3)` gives 0, 3, 6, …, 24,
the start of every codon.

In [ ]:
# ── 2.2 Loops: split the fragment into codons and count bases ──
codons = []
for start in range(0, len(dna), 3):               # 0, 3, 6, ..., 24
    codons.append(dna[start:start + 3])
print("codons:", codons)

n_g = 0
for base in dna:                                  # visit every base once
    if base == "G":
        n_g = n_g + 1
print("G counted by hand:", n_g, "| with dna.count('G'):", dna.count("G"))

squares = [k ** 2 for k in range(5)]              # a 'list comprehension': a loop written in one line
codons_again = [dna[s:s + 3] for s in range(0, len(dna), 3)]
print("list comprehension:", squares, "| same codons?", codons == codons_again)

🔍 **What just happened?** The first loop visited the codon starts 0, 3, …, 24 and collected **9 codons**. The second
loop counted Gs one by one and agrees with the built-in `count` — when you write your own version of something, always
check it against a trusted one. The last lines show a **list comprehension**, `[expression for item in sequence]`: a
compact loop that builds a list.

### 2.3 Conditions: `if`, `elif`, `else`

`if condition:` runs its indented block only when the condition is `True`; `elif` ("else if") tries another condition;
`else` catches everything left. Below we classify bases chemically: **purines** (A, G) have two rings, **pyrimidines**
(C, T) one.

In [ ]:
# ── 2.3 Conditions: purines and pyrimidines ──
for base in "AGCTN":
    if base in "AG":
        kind = "purine (two rings)"
    elif base in "CT":
        kind = "pyrimidine (one ring)"
    else:
        kind = "unknown — 'N' means the sequencer could not decide"
    print(base, "→", kind)

🔍 **What just happened?** Each base went through the `if`/`elif`/`else` chain and only the first matching branch ran.
Real sequence files contain **N** for undetermined bases, so good code always has an `else` for the unexpected.

### 2.4 Dictionaries: look-up tables

A **dictionary** (`dict`) maps **keys** to **values**: `{"A": "T", "T": "A", ...}`. You look a value up with square
brackets, `complement["A"]` → `"T"`, add or change entries with `d[key] = value`, and loop over pairs with
`for key, value in d.items()`. Dictionaries are perfect for counting and for code tables.

In [ ]:
# ── 2.4 Dictionaries: counting bases, base pairing ──
counts = {}                                       # an empty dictionary
for base in dna:
    counts[base] = counts.get(base, 0) + 1        # .get(key, 0): current count, or 0 if not seen yet
print("base counts:", counts)

complement = {"A": "T", "T": "A", "G": "C", "C": "G", "N": "N"}
print("complement of G:", complement["G"])
for base, n in sorted(counts.items()):
    print(f"  {base}: {n:2d} ({n / len(dna):.0%}), pairs with {complement[base]}")

🔍 **What just happened?** `counts` started empty and gained one entry per new base; `counts.get(base, 0)` returned 0
the first time a base was seen. `complement` encodes Watson–Crick pairing. `sorted(...)` put the bases in alphabetical
order for printing.

### 2.5 Functions: name a computation, reuse it, test it

A **function** packages a computation under a name. `def` starts it, the arguments go in parentheses, the indented
body follows, and `return` hands back the result. The string right under `def` is the **docstring**: it explains what
the function does (type `help(gc_content)` to read it). Two classic tools:

* **GC content** — the fraction of G and C bases.
* **Reverse complement** — DNA is double-stranded and the two strands run in opposite directions (antiparallel). To
  read the *other* strand in the usual 5′→3′ direction, complement every base *and* reverse the order. A gene on the
  "minus" strand appears in a genome file as the reverse complement of its sequence.

`seq[::-1]` is a slice with step −1: the string backwards. (The course toolkit has its own `reverse_complement`, which
also handles ambiguity codes such as R and Y; we call our version `revcomp`, a common abbreviation.)

In [ ]:
# ── 2.5 Functions with docstrings, checked with assert ──
def gc_content(seq: str) -> float:
    """Fraction of bases in seq that are G or C (upper- or lower-case)."""
    seq = seq.upper()
    return (seq.count("G") + seq.count("C")) / len(seq)


def revcomp(seq: str) -> str:
    """Reverse complement: the sequence of the opposite DNA strand, read 5'→3' (complement each base, then reverse)."""
    return "".join(complement[base] for base in seq.upper())[::-1]


# assert stops with an error if the statement is False: a tiny automatic test
assert revcomp("ATGC") == "GCAT"
assert revcomp(revcomp(dna)) == dna                            # doing it twice gives back the original
assert gc_content("GGCC") == 1.0 and gc_content("ATAT") == 0.0
print("reverse complement of", dna[:12], "→", revcomp(dna[:12]))
print(f"GC content: {gc_content(dna):.3f}  (lower-case input works too: {gc_content(dna.lower()):.3f})")

🔍 **What just happened?** The functions work, and the `assert` lines prove it on cases we can check by hand: the
reverse complement of ATGC is GCAT, and taking the reverse complement twice returns the original. When an assert
fails, Python stops with an `AssertionError` — a cheap, powerful habit for catching bugs early.
`"".join(list_of_strings)` glues strings together with nothing in between.

### 2.6 The genetic code as a dictionary

The **genetic code** maps each of the $4^3 = 64$ codons to an amino acid (one-letter code) or to "stop" (we write `*`).
Textbooks print it as a 4 × 4 × 4 table ordered T, C, A, G for the first, second and third base. Reading that table row
by row gives a 64-letter string; three nested loops turn it into a dictionary.

In [ ]:
# ── 2.6 Build the standard genetic code (NCBI translation table 1) ──
ORDER = "TCAG"
AMINO = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"   # the textbook table, row by row
genetic_code = {}
k = 0
for first in ORDER:
    for second in ORDER:
        for third in ORDER:
            genetic_code[first + second + third] = AMINO[k]
            k += 1                                   # 'k += 1' is short for 'k = k + 1'

print("number of codons:", len(genetic_code))
print("ATG →", genetic_code["ATG"], "(methionine, the start)   TGG →", genetic_code["TGG"], "(tryptophan)")
print("stop codons:", [c for c, aa in genetic_code.items() if aa == "*"])
print("codons for leucine (L):", [c for c, aa in genetic_code.items() if aa == "L"])

🔍 **What just happened?** 64 codons. ATG codes methionine (M) and TGG tryptophan (W) — the only two amino acids with a
single codon — and the dictionary agrees with the textbook on the three stop codons, TAA, TAG and TGA. Leucine has
**six** codons: the code is *redundant*, so many mutations do not change the protein. Let us use it.

### 2.7 Translating the insulin gene

Insulin is made as a precursor, **preproinsulin** (110 amino acids): a 24-residue **signal peptide** that steers it
into the secretory pathway, the **B chain** (30 aa), a connecting **C-peptide**, and the **A chain** (21 aa). Enzymes cut
out the signal peptide and the C-peptide, and the B and A chains stay linked by disulfide bridges. Below is a
333-nucleotide open reading frame that encodes human preproinsulin; we translate it and compare the result with the
protein sequence stored in UniProt (entry P01308). Both are short public sequences, typed into the cell below as text —
nothing is downloaded.

In [ ]:
# ── 2.7 Translate an open reading frame and find the insulin chains ──
insulin_orf = ("ATGGCCCTGTGGATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGACCCAGCCGCAGCC"
               "TTTGTGAACCAACACCTGTGCGGCTCACACCTGGTGGAAGCTCTCTACCTAGTGTGCGGGGAACGAGGCTTC"
               "TTCTACACACCCAAGACCCGCCGGGAGGCAGAGGACCTGCAGGTGGGGCAGGTGGAGCTGGGCGGGGGCCCT"
               "GGTGCAGGCAGCCTGCAGCCCTTGGCCCTGGAGGGGTCCCTGCAGAAGCGTGGCATTGTGGAACAATGCTGT"
               "ACCAGCATCTGCTCCCTCTACCAGCTGGAGAACTACTGCAACTAG")


def translate(orf: str) -> str:
    """Translate a coding sequence codon by codon, stopping at (and not including) the first stop codon."""
    protein = ""
    for start in range(0, len(orf) - 2, 3):
        aa = genetic_code[orf[start:start + 3]]
        if aa == "*":
            break                                    # 'break' leaves the loop immediately
        protein += aa
    return protein


UNIPROT_P01308 = ("MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN")
preproinsulin = translate(insulin_orf)
print(f"ORF: {len(insulin_orf)} nt = {len(insulin_orf) // 3} codons, GC content {gc_content(insulin_orf):.1%}")
print("protein:", preproinsulin)
print("length:", len(preproinsulin), "aa | identical to UniProt P01308:", preproinsulin == UNIPROT_P01308)
B_chain, A_chain = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT", "GIVEQCCTSICSLYQLENYCN"
print("B chain starts at residue", preproinsulin.find(B_chain) + 1, "| A chain starts at residue", preproinsulin.find(A_chain) + 1)
print("signal peptide:", preproinsulin[:24])

🔍 **What just happened?** 333 nucleotides = 111 codons; the last one (TAG) is the stop, so the protein has **110
amino acids** and matches UniProt exactly. `str.find` returned the position of each chain (Python index + 1 = the
biologist's residue number): the **B chain starts at residue 25**, right after the 24-residue signal peptide
`MALWMRLLPLLALLALWGPDPAAA` (notice how hydrophobic — L, A, W — a typical signal peptide is), and the **A chain at
residue 90**. The insulin ORF is GC-rich (64.6 %).

### 2.8 Point mutations: synonymous, missense, nonsense

A **point mutation** changes one base. Because the code is redundant, it can be **synonymous** (same amino acid),
**missense** (a different amino acid) or **nonsense** (a premature stop codon, truncating the protein).

In [ ]:
# ── 2.8 A tiny mutation explorer ──
def mutate(seq: str, position: int, new_base: str) -> str:
    """Return a copy of seq with the base at 1-based `position` replaced by new_base."""
    i = position - 1                                  # biologists count from 1, Python from 0
    return seq[:i] + new_base + seq[i + 1:]


def classify_mutation(orf: str, position: int, new_base: str) -> str:
    """Compare the proteins encoded by orf before and after a point mutation."""
    before, after = translate(orf), translate(mutate(orf, position, new_base))
    if after == before:
        return "synonymous (protein unchanged)"
    if len(after) < len(before):
        return f"nonsense: protein truncated to {len(after)} aa"
    diff = next(k for k in range(len(before)) if before[k] != after[k])   # index of the first residue that differs
    return f"missense: residue {diff + 1} {before[diff]} → {after[diff]}"


for pos, base in [(6, "T"), (12, "A"), (31, "T"), (5, "T")]:
    old = insulin_orf[pos - 1]
    print(f"base {pos:3d} {old}→{base} (codon {(pos - 1) // 3 + 1}): {classify_mutation(insulin_orf, pos, base)}")

🔍 **What just happened?** Four single-base changes, three kinds of outcome. Changing base 6 (the third base of
codon 2, GCC → GCT) keeps alanine: **synonymous** — third codon positions are often "wobble" positions where the code
tolerates change. Base 12 (TGG → TGA, codon 4) turns tryptophan into a **stop**: the protein is cut after 3 residues —
a **nonsense** mutation. Base 31 changes the *first* base of codon 11 (CTG → TTG), yet the protein is unchanged: both
codons mean leucine, one of the six-codon amino acids. Base 5 (GCC → GTC) is **missense**: alanine 2 becomes valine.
Real mutations in *INS* cause forms of neonatal diabetes.

✏️ **Your turn.** (1) Mutate base 1 (`A` → `G`). Our function reports a missense change, M → V at residue 1 — but is
that what a cell would do? Without its ATG start codon, a human ribosome would usually begin at a later ATG or make
no insulin at all: code only knows the rules we gave it. (2) Find a
mutation in the B chain that is synonymous and one that is missense. (3) Use `revcomp(insulin_orf)` and
check: does the *other* strand also start with ATG?

### 📘 Lecture notes — the programming prerequisite of 02-620

**What 02-620 assumes.** Basic linear algebra, probability and statistics (CMU 02-680) and programming (CMU 02-601):
*basic familiarity with Python and NumPy*, being *able to do simple linear algebra using NumPy*, and being able to
*save and open Jupyter notebooks*. The recitations begin with reviews of probability, linear algebra and coding.
Concretely, by the first homework you should be able to do everything in this module without looking it up.

**The core of Python in one table.**

| Idea | Syntax | Example from this module |
|---|---|---|
| variable, assignment | `name = value` | `dna = "ATGGCC..."` |
| types | `int`, `float`, `str`, `bool` | `20 / 3` is a float |
| index from 0, slice excludes end | `s[0]`, `s[a:b]`, `s[::-1]` | `dna[0:3]` = first codon |
| list, dictionary | `[...]`, `{key: value}` | stop codons; the genetic code |
| loop, condition | `for x in seq:` / `if ...: elif ...: else:` | codons; purines vs pyrimidines |
| function + test | `def f(x):` + docstring + `return ...`; `assert` | `gc_content`, `revcomp`, `translate` |

**A formula turned into code.** The GC content of a sequence $s$ of length $L$ is

$$\text{GC}(s) = \frac{n_G(s) + n_C(s)}{L}$$

*In words:* count the Gs and the Cs, add them, divide by the length. In Python:
`(seq.count("G") + seq.count("C")) / len(seq)`. *Worked example:* the first 27 bases of the insulin coding sequence
contain 19 G or C, so GC = 19/27 = 0.704.

**Common confusions.** `=` assigns, `==` compares; Python counts from 0 while biologists count bases from 1;
indentation is part of the syntax; a function that is never *called* computes nothing.

> 🎓 **Exam tip.** The homework assumes you can turn a formula into NumPy code quickly. Practise translating loops into array
    operations (Section 3.6) — exam questions often show a short code snippet and ask what it computes or what shape it
    returns.

---
## 3 · NumPy arrays: the genotype matrix

> 📍 **Where we are.** Lists are flexible but slow and clumsy for tables of numbers. Every method in 02-620 operates
> on a **matrix** X with N rows (samples) and J columns (features) — Module 00, Section 3. **NumPy** is the Python
> library for such arrays.

A NumPy **array** is a grid of numbers of one type. Its **shape** says how many entries it has along each axis: a
genotype matrix of 200 people × 1,000 SNPs has shape `(200, 1000)`; **axis 0** runs down the rows (people), **axis 1**
across the columns (SNPs). Its **dtype** says what kind of number it stores (`int8` = small integers using 1 byte,
`float64` = decimals using 8 bytes).

**Genotype coding.** At a SNP with alleles A and T, a diploid person is AA, AT or TT. As in the 02-620 regression
lecture (Module 06) we store the number of copies of one chosen allele: **AA = 0, AT = 1, TT = 2**. The toolkit's
`simulate_genotypes` draws each SNP's allele frequency between 0.05 and 0.5 and assumes Hardy–Weinberg equilibrium
(Module 02 explains what that means).

In [ ]:
# ── 3.1 Simulate a genotype matrix: N = 200 individuals × J = 1,000 SNPs ──
rng = np.random.default_rng(SEED + 3)                  # re-seed so that this section always gives the same matrix
G, pop, freqs = simulate_genotypes(rng, n_samples=200, n_snps=1000)
print("type:", type(G))
print("shape:", G.shape, "→ N =", G.shape[0], "individuals, J =", G.shape[1], "SNPs")
print("dtype:", G.dtype, "| memory:", G.nbytes, "bytes | number of dimensions:", G.ndim)
print("the first 5 individuals at the first 8 SNPs:")
print(G[:5, :8])

🔍 **What just happened?** `G` is a 200 × 1,000 array of small integers — **200,000 genotypes** in 200,000 bytes (one
byte each, because the dtype is `int8`). Each row is one person, each column one SNP; every entry is 0, 1 or 2.
`simulate_genotypes` also returned `freqs`, the *true* allele frequency of every SNP (shape 1 × 1,000), which we will
try to recover from the data.

### 3.2 Indexing and slicing in two dimensions

`G[i, j]` is the genotype of person *i* at SNP *j*. A colon means "everything along this axis": `G[i]` or `G[i, :]` is
row *i* (one person, all SNPs), `G[:, j]` is column *j* (one SNP, all people), and `G[10:15, 100:105]` is a 5 × 5 block.

In [ ]:
# ── 3.2 Indexing and slicing ──
print("person 0, SNP 0:", G[0, 0])
print("person 0, all SNPs → shape", G[0].shape, "; first ten:", G[0, :10])
print("SNP 0, all people  → shape", G[:, 0].shape, "; first ten:", G[:10, 0])
block = G[10:15, 100:105]
print("a 5 × 5 block, shape", block.shape)
print(block)
print("last person, last SNP:", G[-1, -1])

🔍 **What just happened?** A single row has shape `(1000,)` — a one-dimensional array of 1,000 numbers — and a single
column has shape `(200,)`. Slicing two axes at once gives a smaller 2-D array. Negative indices count from the end, as
with strings.

### 3.3 Boolean masks: selecting by a condition

Comparing an array with a value gives an array of `True`/`False` of the same shape — a **mask**. Masks can be counted
(`True` = 1), and used to *select* rows or columns: `G[mask]` keeps the rows where the mask is `True`.

In [ ]:
# ── 3.3 Boolean masks ──
hom_snp0 = G[:, 0] == 2                                 # who carries two copies at SNP 0?
print("mask:", hom_snp0[:10], "... shape", hom_snp0.shape)
print("number of homozygotes (2) at SNP 0:", hom_snp0.sum(), "→ their row numbers:", np.flatnonzero(hom_snp0)[:8], "...")
print("their genotypes at SNPs 0–5:")
print(G[hom_snp0][:4, :6])
rare = (G == 2).sum(axis=0) == 0                        # SNPs where nobody is homozygous for the counted allele
print("SNPs with no '2' genotype at all:", rare.sum(), "of", G.shape[1])

🔍 **What just happened?** `G[:, 0] == 2` compared all 200 genotypes of SNP 0 with 2 at once; summing the mask counted
the homozygotes, and `np.flatnonzero` listed their row numbers. `G[hom_snp0]` kept only those people (every column).
The last line applied a condition to the whole matrix, counted the `True`s down each column, and found the SNPs where
nobody carries two copies — rare alleles (frequency near 0.05) often show no homozygotes in 200 people.

### 3.4 Reductions along an axis: allele frequencies

🧮 **Math refresher — the mean and the summation sign.** The **mean** (average) of numbers $x_1, x_2, \dots, x_N$ is

$$\bar x = \frac{1}{N}\sum_{i=1}^{N} x_i = \frac{x_1 + x_2 + \dots + x_N}{N}.$$

*In words:* add all the numbers and divide by how many there are. The symbol $\sum_{i=1}^{N}$ (capital Greek sigma)
means "add up the following expression for $i = 1, 2, \dots, N$"; $\bar x$ is read "x bar".

The **allele frequency** of SNP *j* — the fraction of all 2N chromosomes carrying the counted allele — is
$\hat p_j = \frac{1}{2N}\sum_{i=1}^{N} G_{ij}$, the mean genotype divided by 2 (each person carries two copies of the
chromosome). The hat on $\hat p$ marks an **estimate** computed from data. In NumPy, `G.mean(axis=0)` averages *down*
axis 0 — over people — leaving one number per SNP; `axis=1` would average across SNPs, one number per person.

In [ ]:
# ── 3.4 Allele frequencies with axis=0, per-person totals with axis=1 ──
p_hat = G.mean(axis=0) / 2                              # one estimate per SNP
alleles_per_person = G.sum(axis=1)                      # one total per person
print("p_hat has shape", p_hat.shape, "| first five:", p_hat[:5].round(3))
print("true freqs (first five):        ", freqs[0, :5].round(3))
error = p_hat - freqs[0]
print(f"typical estimation error: {np.abs(error).mean():.3f}  (largest {np.abs(error).max():.3f})")
print("alleles_per_person has shape", alleles_per_person.shape, "| min", alleles_per_person.min(),
      "| mean", alleles_per_person.mean(), "| max", alleles_per_person.max())

🔍 **What just happened?** `G.mean(axis=0)` collapsed the 200 rows into one row of 1,000 means, so `p_hat` has shape
`(1000,)`. The estimates are close to the true frequencies but not identical: with 400 chromosomes per SNP the
average error is a couple of percentage points (printed above). Sampling error shrinks as N grows — Module 02 makes
this precise. `G.sum(axis=1)` gave one number per person: the total count of the chosen alleles they carry across the
1,000 SNPs.

⚠️ **Which allele is "counted"?** In 02-620 the regression lecture codes AA = 0, AT = 1, TT = 2 (it counts T), while
the naïve-Bayes example (Module 08) calls AA "minor-allele homozygous" — that is, it counts the *major* allele. Both are fine: the genotype $G$ and $2 - G$ carry the
same information, only the direction flips. The toolkit counts copies of the allele with frequency ≤ 0.5 (the minor
allele). Always check the convention before interpreting a positive or negative effect.

### 3.5 Broadcasting: centring every column at once

Subtracting each SNP's mean from its column is a step in almost every method of the course (regression, PCA). NumPy
lets you write `G - G.mean(axis=0)`: a `(200, 1000)` array minus a `(1000,)` array. The smaller array is
**broadcast** — conceptually copied down all 200 rows — so that each column loses its own mean. The rule: shapes are
compared from the right, and a dimension of size 1 (or a missing one) is stretched to match.

In [ ]:
# ── 3.5 Broadcasting ──
col_means = G.mean(axis=0)                              # shape (1000,)
G_centred = G - col_means                               # (200, 1000) − (1000,) → (200, 1000)
print("shapes:", G.shape, "-", col_means.shape, "→", G_centred.shape, "| dtype now:", G_centred.dtype)
print("column means after centring: largest |mean| =", f"{np.abs(G_centred.mean(axis=0)).max():.1e}")
row_means = G.mean(axis=1, keepdims=True)               # shape (200, 1): keepdims keeps it as a column
print("subtracting per-person means:", G.shape, "-", row_means.shape, "→", (G - row_means).shape)
Z, mu, sd = standardize(G)                              # the toolkit does centring AND scaling
print(f"standardised: column means ≈ {np.abs(Z.mean(axis=0)).max():.1e}, column sds ≈ {Z.std(axis=0).mean():.3f}")

🔍 **What just happened?** After broadcasting, every column mean is 0 up to rounding errors around $10^{-16}$ — the
computer's tiny floating-point fuzz, which you should read as "exactly zero". The dtype became `float64`, because means
are decimals. To subtract a value *per row*, the means must have shape `(200, 1)` (a column), which `keepdims=True`
provides. The toolkit's `standardize` also divides each column by its standard deviation, giving **z-scores** (mean 0,
sd 1) — Module 00 used them for the two-gene classifier.

### 3.6 Vectorisation: why NumPy instead of loops

The same allele frequencies could be computed with two nested Python loops. NumPy's `G.mean(axis=0)` runs the loop in
compiled C code instead. How much faster is it?

In [ ]:
# ── 3.6 Loops versus vectorised NumPy ──
def allele_freq_loop(genotypes: np.ndarray) -> list:
    """Allele frequency of every column, computed with plain Python loops (slow but transparent)."""
    n_people, n_snps = genotypes.shape
    result = []
    for j in range(n_snps):
        total = 0
        for i in range(n_people):
            total += int(genotypes[i, j])           # int(...): add as a Python integer (see Section 7)
        result.append(total / (2 * n_people))
    return result


t0 = time.perf_counter(); f_loop = allele_freq_loop(G); t_loop = time.perf_counter() - t0
t_numpy = []
for _ in range(5):                                    # a NumPy call is so fast that we time it 5 times, keep the best
    t0 = time.perf_counter(); f_numpy = G.mean(axis=0) / 2; t_numpy.append(time.perf_counter() - t0)
t_numpy = min(t_numpy)
print(f"Python loops: {t_loop * 1000:7.1f} ms")
print(f"NumPy:        {t_numpy * 1000:7.3f} ms   → about {t_loop / t_numpy:,.0f}× faster on this computer")
print("same answer:", np.allclose(f_loop, f_numpy))
scale = (10_000 * 500_000) / G.size                   # a chip of 500,000 SNPs typed in 10,000 people
print(f"extrapolated to 10,000 people × 500,000 SNPs ({scale:,.0f}× more genotypes): "
      f"NumPy ≈ {t_numpy * scale:,.0f} s, loops ≈ {t_loop * scale / 60:,.0f} min")

🔍 **What just happened?** Both versions give the same frequencies, but the loop takes tens of milliseconds while NumPy
takes a fraction of a millisecond: NumPy is tens to hundreds of times faster, depending on the computer — the cell
prints the factor on yours (timings of such short operations also vary from run to run). A real chip with 500,000
SNPs typed in 10,000 people has 25,000 times more genotypes; the last line scales both timings up: seconds with NumPy
against many minutes with loops — and a real analysis repeats such steps many times. **Vectorised**
code — whole-array operations instead of explicit loops — is also shorter and closer to the mathematics.

✏️ **Your turn.** The **observed heterozygosity** of a SNP is the fraction of people with genotype 1. Compute it for all
1,000 SNPs in *one line* with a mask and `axis=0` (hint: `(G == 1).mean(axis=0)`). Which SNP has the highest value, and
what is its allele frequency? (`np.argmax` returns the position of the largest entry.) You will need this quantity in
Section 8.

### 📘 Lecture notes — the data matrix of genetic association studies

**Setting (from the regression lecture on).** N individuals are measured for a **phenotype** — a vector
$\mathbf y = (y_1, \dots, y_N)$, e.g. the expression of the *BRCA1* gene, blood pressure or insulin level — and
genotyped at J SNPs, giving an **N × J genotype matrix** $X$ with entries

$$X_{ij} \in \{0, 1, 2\} = \text{copies of the counted allele carried by individual } i \text{ at SNP } j$$

*In words:* row $i$ is a person, column $j$ is a SNP, and each entry counts alleles (AA = 0, AT = 1, TT = 2).
*Univariate* regression relates $\mathbf y$ to one column at a time (one SNP); *multivariate* regression uses all
columns at once, to estimate the effect of each SNP in the presence of all the other SNPs (Module 06).

**Two column operations used everywhere.** The allele frequency and the centred matrix:

$$\hat p_j = \frac{1}{2N}\sum_{i=1}^{N} X_{ij}, \qquad \tilde X_{ij} = X_{ij} - \bar x_j \;\;\text{with}\;\; \bar x_j = \frac{1}{N}\sum_{i=1}^{N} X_{ij}$$

*In words:* the allele frequency is the column mean divided by 2 (two chromosomes per person); centring subtracts
each column's mean from every entry of that column, so every column then averages to zero. In NumPy:
`X.mean(axis=0) / 2` and `X - X.mean(axis=0)` (broadcasting).

*Worked example:* for four people at one SNP with genotypes 0, 1, 1, 2, the column mean is 1, so
$\hat p = 4/8 = 0.5$ and the centred column is −1, 0, 0, 1.

> 🎓 **Exam tip.** Keep the orientation straight: rows = individuals (N), columns = SNPs (J). `X.mean(axis=0)` is one value
    per SNP, `X.mean(axis=1)` one per person; `X @ w` (Module 03) needs w to have J entries.

---
## 4 · Random numbers and reproducibility

> 📍 **Where we are.** You just used `simulate_genotypes`, which relies on random numbers. Simulation is how this
> course tests every method on data where we *know* the truth. Here is how computer randomness works.

Computers produce **pseudo-random** numbers: a deterministic algorithm generates a sequence that looks random. The
sequence is fixed by a starting value, the **seed**. In NumPy you create a **generator** with
`rng = np.random.default_rng(seed)` and ask it for numbers: `rng.random(n)` (uniform between 0 and 1),
`rng.integers(a, b, n)`, `rng.normal(mean, sd, n)`, `rng.binomial(n_trials, p, size)`, `rng.choice(options, n)`. Every
notebook of this course sets `SEED = 0` in its setup cell, so that your numbers match the text.

In [ ]:
# ── 4.1 Same seed → same numbers; different seed → different numbers ──
print("seed 42:      ", np.random.default_rng(42).random(4).round(3))
print("seed 42 again:", np.random.default_rng(42).random(4).round(3))
print("seed 7:       ", np.random.default_rng(7).random(4).round(3))
g = np.random.default_rng(42)
print("one generator, two calls:", g.random(2).round(3), g.random(2).round(3), "← continues the same sequence")

🔍 **What just happened?** The same seed reproduced exactly the same four numbers; a different seed gave different
ones. One generator called twice *continues* its sequence — so the numbers a cell gets depend on how many random
numbers earlier cells used. That is why sections that must give fixed results start with
`rng = np.random.default_rng(SEED + k)`.

### 4.2 Coin flips and Mendel's alleles

A fair coin lands heads with probability 0.5. Draw a uniform number $u$ between 0 and 1: $u < 0.5$ happens half the
time, so `rng.random(n) < 0.5` simulates *n* coin flips. A diploid genotype is two such "coin flips" — one allele from
each parent — where the coin shows the counted allele with probability $p$ (the allele frequency). The number of
counted alleles, 0, 1 or 2, is then a **binomial** count, which NumPy can also draw directly with
`rng.binomial(2, p, size)`.

In [ ]:
# ── 4.2 Coin flips, and genotypes from two allele draws ──
rng = np.random.default_rng(SEED + 4)
for n in [10, 100, 10_000]:
    heads = rng.random(n) < 0.5
    print(f"{n:6d} flips: fraction of heads = {heads.mean():.3f}")

p = 0.3                                                # frequency of the counted allele
n_people = 10_000
from_mother = rng.random(n_people) < p                 # True = the counted allele
from_father = rng.random(n_people) < p
genotype = from_mother.astype(int) + from_father.astype(int)     # 0, 1 or 2 copies
direct = rng.binomial(2, p, n_people)                   # the same distribution, drawn directly
for g_value, mendel in zip([0, 1, 2], [(1 - p) ** 2, 2 * p * (1 - p), p ** 2]):
    print(f"genotype {g_value}: two draws {np.mean(genotype == g_value):.3f} | binomial {np.mean(direct == g_value):.3f}"
          f" | (1−p)², 2p(1−p), p² → {mendel:.3f}")

🔍 **What just happened?** With 10 flips the fraction of heads can be far from 0.5; with 10,000 it is very close — a
first glimpse of the **law of large numbers** (Module 02). For genotypes, combining two independent allele draws and
drawing a binomial count give the same frequencies, close to $(1-p)^2 = 0.49$, $2p(1-p) = 0.42$ and $p^2 = 0.09$ for
$p = 0.3$. These are the **Hardy–Weinberg proportions**: the chance that *both* alleles are the counted one is
$p \times p$, that neither is $(1-p)^2$, and a heterozygote can arise in two ways (mother or father contributes the
counted allele), hence $2p(1-p)$. Module 02 derives this from the rules of probability; Section 8 tests it on data.

⚠️ **Seeds and honesty.** A seed makes results reproducible — someone else can rerun your notebook and get your numbers.
It must never be *tuned*: trying seeds until a simulation "works" is a form of cherry-picking. A conclusion that holds
for only one seed is not a conclusion. (Try changing `SEED + 4` above: the numbers move a little, the message doesn't.)

---
## 5 · pandas: the leukaemia data as a table

> 📍 **Where we are.** NumPy arrays hold numbers only. Real tables also have **names** — of samples, of genes — and
> mixed columns (a text label next to numbers). **pandas** adds exactly that: the `DataFrame`, a table with labelled
> rows and columns.

We generate the leukaemia data of Module 00 (72 patients × 3,571 genes, log10 expression, 47 ALL and 25 AML) and build
a small DataFrame with the diagnosis and six marker genes: three that are higher in AML (ZYX, CD33, CST3) and three
that are higher in ALL (MYB, CD79A, TCF3), as in the original study of Golub *et al.*

> 🧪 **Why simulated?** The course stores no data sets: every table is generated by code, so each notebook runs on its
> own. And simulated data come with the truth — the simulator records which genes really differ — so we can check
> what our analysis finds. The numbers are *not* those of the original study.

In [ ]:
# ── 5.1 A DataFrame with the diagnosis and six named genes ──
leuk = simulate_leukemia_expression()
print("Source:", leuk["source"])
col = {g: j for j, g in enumerate(leuk["genes"])}                # gene symbol → column number
SIX_GENES = ["ZYX", "CD33", "CST3", "MYB", "CD79A", "TCF3"]       # 3 AML-high, then 3 ALL-high markers
expr = pd.DataFrame({"label": leuk["labels"]}, index=pd.Index(leuk["sample_id"], name="patient"))
for g in SIX_GENES:
    expr[g] = leuk["X"][:, col[g]]                               # one new column per gene
print("shape:", expr.shape)
print(expr.head())

🔍 **What just happened?** `expr` is a table with **72 rows** (patients, labelled by their sample number) and **7
columns** (the label and six genes). `head()` shows the first five rows. Unlike a NumPy array, every row and column has
a name, so you can write `expr["ZYX"]` instead of remembering that ZYX is column 1. (Patient 1's CD79A value, 4.2041,
is the ceiling of the measurement scale, about 16,000 units.)

Some essential operations:

| Code | What it does |
|---|---|
| `df["ZYX"]` | one column (a `Series`) |
| `df[["ZYX", "CD33"]]` | several columns (a smaller DataFrame) |
| `df[df["label"] == "AML"]` | the rows where a condition holds (a boolean mask, as in NumPy) |
| `df.loc[5, "ZYX"]` | one value by row and column *names* |
| `df.describe()` | summary statistics of every numeric column |
| `df.groupby("label").mean()` | the mean of every column within each group |
| `df.to_numpy()` | back to a plain NumPy array |

In [ ]:
# ── 5.2 Selecting and summarising ──
print(expr.describe().round(2))
aml = expr[expr["label"] == "AML"]
print(f"\nAML patients: {len(aml)}; ZYX in patient 30: {expr.loc[30, 'ZYX']:.2f}")
print("\nmean expression per class (log10 units):")
by_class = expr.groupby("label").mean()
print(by_class.round(2))
print("\nALL − AML difference:")
print((by_class.loc["ALL"] - by_class.loc["AML"]).round(2))

🔍 **What just happened?** `describe()` reports, for each gene, the count, mean, standard deviation, minimum,
quartiles and maximum over all 72 patients (the text column `label` is skipped). Notice the maxima: four genes reach
4.20, the ceiling of the scale, and MYB's minimum is 2.00, the floor. `groupby("label").mean()` split the patients
into ALL and AML and averaged each gene within each group: the three AML-high genes have **negative** ALL − AML
differences (−0.47 to −0.78) and the three ALL-high genes **positive** ones (0.53 to 0.83). A difference of 1 log10
unit means a **ten-fold** difference in expression; 0.78 means $10^{0.78} \approx 6$-fold.

### 5.3 Ranking all 3,571 genes

Now the same idea for every gene: put the whole matrix in a DataFrame, average within classes, and sort the genes by
the absolute difference of the class means.

In [ ]:
# ── 5.3 The ten genes with the largest ALL–AML mean difference ──
full = pd.DataFrame(leuk["X"], columns=leuk["genes"])
class_means = full.groupby(leuk["labels"]).mean()             # 2 rows (ALL, AML) × 3,571 columns
difference = class_means.loc["ALL"] - class_means.loc["AML"]
top10 = difference.abs().sort_values(ascending=False).head(10)
table = pd.DataFrame({"ALL − AML (log10)": difference[top10.index].round(2),
                      "fold change": (10 ** difference[top10.index].abs()).round(1),
                      "higher in": np.where(difference[top10.index] > 0, "ALL", "AML"),
                      "protein": [LEUKEMIA_MARKERS.get(g, ("", "—"))[1] for g in top10.index]})
print(table)

🔍 **What just happened?** `groupby` worked on all 3,571 columns at once and produced a 2 × 3,571 table of class
means; one subtraction and one sort later we have the ten genes with the largest mean difference, between 6-fold and
10-fold (the "fold change" column; $10^{1.00} = 10$). All ten carry marker names. Two are old friends of
haematologists: **DNTT** encodes terminal deoxynucleotidyl transferase (TdT), a classic marker of lymphoblasts (higher
in ALL), and **MPO** encodes myeloperoxidase, the enzyme pathologists stain for to recognise myeloid blasts (higher in
AML). Module 00 ranked genes by Golub's *signal-to-noise* score instead, which divides the difference by the spread
within classes; a big difference in a very variable gene can be less useful than a moderate difference in a stable one.

Because the data are simulated, we can ask the simulator *why* these genes differ.

In [ ]:
# ── 5.4 🧪 Ground-truth check: which of the top genes have a true ALL-vs-AML effect? ──
truth = pd.Series(leuk["true_effect"], index=leuk["genes"])         # simulated AML − ALL effect (0 = none)
print(pd.DataFrame({"observed ALL − AML": difference[top10.index].round(2),
                    "true ALL − AML effect": (-truth[top10.index]).round(2)}))
print("\nCD79A by subtype (log10 means):", full["CD79A"].groupby(leuk["subtype"]).mean().round(2).to_dict())

🔍 **What just happened?** For six of the ten genes the observed difference matches a true ALL-vs-AML effect in the
simulation (e.g. DNTT: observed 1.00, true 0.95). But four — **CD19, PAX5, CD79A, CD22** — have a true ALL-vs-AML
effect of **zero**, yet differ by almost a log10 unit! The last line explains it: these are **B-cell genes**. CD79A
averages 3.98 in B-ALL, 2.52 in T-ALL and 2.87 in AML. Because 38 of the 47 ALL patients are B-ALL, the ALL average is
pulled up. The difference is real, but it belongs to a **subgroup**, not to ALL as a whole — a T-ALL patient does not
show it. Group averages can hide subgroups; always look at the distribution, not just the mean (Section 6).

⚠️ **pandas aligns by labels.** When you combine two Series, pandas matches them by their index labels, not by
position. This is usually what you want (above, `difference` and `truth` were matched gene by gene), but it surprises
people who expect NumPy-style position-by-position arithmetic. Use `.to_numpy()` when you mean positions.

✏️ **Your turn.** Sort by the *signed* difference instead (`difference.sort_values()`) and print the five most
AML-high and the five most ALL-high genes. Then compute the difference of class **medians** (`.median()` instead of
`.mean()`). Do the top genes change?

### 📘 Lecture notes — expression profiles as inputs X, diagnoses as outputs Y

**The case study.** The introductory lecture of 02-620 presents the leukaemia study of Golub *et al.* (1999):
expression of thousands of genes measured by microarrays in patients with acute lymphoblastic (ALL) or acute
myeloid (AML) leukaemia, displayed as a heatmap with **genes as rows and patients as columns** — ALL patients on the
left, AML on the right, red = high, blue = low. In machine-learning terms each patient's **expression profile is
the input X** and the **diagnosis marked above each column is the output Y**.

**In this course's convention** the same data form an N × J matrix with *patients as rows* — the transpose of the
heatmap — and a label vector:

$$X \in \mathbb{R}^{N \times J}, \qquad y_i \in \{0, 1\} \;(0 = \text{ALL},\, 1 = \text{AML})$$

*In words:* N = 72 rows of J = 3,571 real numbers (log10 expression), and one 0/1 label per row.

**Class means and fold change.** For gene $j$ and class $c$ with $N_c$ patients,

$$\bar x_j^{(c)} = \frac{1}{N_c} \sum_{i:\, y_i = c} X_{ij}, \qquad \text{fold change}_j = 10^{\,|\bar x_j^{(\text{ALL})} - \bar x_j^{(\text{AML})}|}$$

*In words:* average the gene over the patients of one class only; on a log10 scale, a difference of $d$ between
the class means is a $10^d$-fold difference in expression. In pandas: `df.groupby(labels).mean()`.
*Worked example (Section 5.3):* DNTT differs by 1.00 log10 unit → a 10-fold difference.

> 🎓 **Exam tip.** When a figure or file has genes as rows, transpose before applying methods that expect samples as rows
    (`df.T` in pandas, `X.T` in NumPy). Mixing up orientation is one of the commonest homework bugs.

---
## 6 · Matplotlib: four plots every biologist needs

> 📍 **Where we are.** Numbers in tables are hard to grasp; pictures are easy. **Matplotlib** is Python's standard
> plotting library. Its pattern is always the same: `fig, ax = plt.subplots()` creates a figure with one or more
> **axes** (panels); you draw on `ax`, label it, and show it.

Every plot in this course has labelled axes *with units*, a title that says what to look at, and a legend only when
needed. Below: a **histogram** (the distribution of one variable), a **scatter plot** (two variables), a **heatmap** (a
whole matrix) and a **line plot** (something changing along an axis).

In [ ]:
# ── 6.1 Histogram and scatter plot of (SIMULATED) expression data ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
bins = np.linspace(expr["ZYX"].min(), expr["ZYX"].max(), 16)
for c, name in enumerate(["ALL", "AML"]):
    values = expr.loc[expr["label"] == name, "ZYX"]
    ax1.hist(values, bins=bins, alpha=0.6, color=CLASS_COLORS[c], edgecolor="k", label=f"{name} (n = {len(values)})")
ax1.set_xlabel("ZYX expression (log10 units)"); ax1.set_ylabel("number of patients")
ax1.set_title("histogram: SIMULATED ZYX is higher in AML on average"); ax1.legend()

for c, name in enumerate(["ALL", "AML"]):
    sub = expr[expr["label"] == name]
    ax2.scatter(sub["CD33"], sub["CD79A"], s=25, color=CLASS_COLORS[c], edgecolor="k", lw=0.3, label=name)
t_all = leuk["subtype"] == "T-ALL"                                # 🧪 the simulator knows each patient's subtype
ax2.scatter(expr["CD33"][t_all], expr["CD79A"][t_all], s=90, facecolor="none", edgecolor="tab:green", lw=1.3,
            label="T-ALL (ringed)")
ax2.set_xlabel("CD33 expression (log10)"); ax2.set_ylabel("CD79A expression (log10)")
ax2.set_title("scatter: two marker genes separate the classes"); ax2.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("patients at the ZYX ceiling (4.2):", expr.groupby("label")["ZYX"].apply(lambda v: int((v >= 4.2).sum())).to_dict())

🔍 **What just happened?** *Left:* the two histograms overlap a lot — ZYX alone is a weak marker in these data — but
look at the tall AML bar at the far right: **17 of the 25 AML patients** sit at 4.2, the ceiling of about 16,000 units
imposed by the preprocessing (Module 00). Their true expression may be higher; the measurement **saturated**. A floor
or ceiling piles values up at one point, which a histogram reveals at a glance and a mean hides. *Right:* the myeloid
marker **CD33** (x) and the B-cell marker **CD79A** (y) place AML patients to the right and most ALL patients towards
the top left. The ALL patients with low CD79A are exactly the **T-ALL** patients (ringed in green, a 🧪 check the
simulation allows): CD79A is a *B-cell* gene, and the ALL group contains both B-lineage and T-lineage leukaemias —
the subgroup we discovered in Section 5.4.

### 6.2 A heatmap with `imshow`

In [ ]:
# ── 6.2 Heatmap of the top-10 genes (rows) × 72 patients (columns) ──
order_patients = np.argsort(leuk["y"], kind="stable")         # ALL first, then AML
block = standardize(full[top10.index].to_numpy())[0][order_patients].T   # genes × patients, z-scores
fig, ax = plt.subplots(figsize=(10, 3.4))
im = ax.imshow(block, aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5)
ax.set_yticks(range(10)); ax.set_yticklabels([f"{g} · {p}" for g, p in zip(top10.index, table["protein"])], fontsize=7)
n_all = int((leuk["y"] == 0).sum())
ax.axvline(n_all - 0.5, color="k"); ax.set_xticks([n_all / 2, n_all + (72 - n_all) / 2]); ax.set_xticklabels(["ALL", "AML"])
ax.set_xlabel("patients"); fig.colorbar(im, ax=ax, label="z-score")
ax.set_title("heatmap: the ten genes with the largest class difference (SIMULATED data)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** `imshow` draws a matrix as coloured squares; each row is one gene converted to z-scores,
each column one patient. Most rows are red on the ALL side and blue on the AML side (ALL-high genes) or the reverse
(MPO, FAH, CST3). The rows of the four B-cell genes (CD19, PAX5, CD79A, CD22) show scattered blue columns *inside*
the ALL block: those are the nine T-ALL patients, which these B-cell genes do not mark.
Arguments worth remembering: `aspect="auto"` stretches the squares to fill the panel, `cmap` chooses the colour map
(`"RdBu_r"`: red high, blue low), and `vmin`/`vmax` fix the colour range so that one extreme value cannot wash out the
picture.

### 6.3 A line plot: watching an estimate settle

In [ ]:
# ── 6.3 Line plot: the running fraction of heads in 2,000 coin flips ──
rng = np.random.default_rng(SEED + 6)
flips = rng.random(2000) < 0.5
running = np.cumsum(flips) / np.arange(1, len(flips) + 1)      # fraction of heads after 1, 2, ..., 2000 flips
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(np.arange(1, len(flips) + 1), running, color="tab:blue", lw=1.2, label="fraction of heads so far")
ax.axhline(0.5, color="k", ls="--", lw=1, label="true probability 0.5")
ax.set_xscale("log"); ax.set_ylim(0, 1)
ax.set_xlabel("number of flips (log scale)"); ax.set_ylabel("fraction of heads")
ax.set_title("line plot: the estimate wanders, then settles near 0.5"); ax.legend()
plt.tight_layout(); plt.show()
print(f"after 10 flips: {running[9]:.2f} | after 100: {running[99]:.3f} | after 2,000: {running[-1]:.3f}")

🔍 **What just happened?** `np.cumsum` gives running totals, so `running[k−1]` is the fraction of heads among the
first *k* flips. Early on the line jumps wildly; later it settles close to 0.5. The **logarithmic x-axis** spreads out
the early flips where the action is. This picture — estimates becoming reliable as data accumulate — is the heart of
statistics and returns in every module.

✏️ **Your turn.** In cell 6.1 replace `"ZYX"` by `"MYB"` in the histogram. Which class is higher now, and where do
the patients at the floor (2.0) sit? Then make your own scatter plot of `"CST3"` against `"TCF3"`, with axis labels
and a title that says what you see.

### 📘 Lecture notes — why 02-620 uses Python, and the tools of this course

**All homework in Python.** 02-620 homework combines written and programming assignments, and the programming
must be done in Python. The reasons given in the course:

- **R** is the statisticians' language, with a huge collection of statistical packages — but it is awkward for
  writing large software;
- **MATLAB** is built for matrix and numerical computation — but it is commercial, resource-hungry and also
  awkward for large programs;
- **Python** is free, open source and lightweight, handles numerical and matrix computation well, supports software
  engineering *and* deep learning — so it has become the main language of machine learning and data science.

**The scientific Python stack used in every module.**

| Library | Object | Typical use here |
|---|---|---|
| NumPy | `ndarray` (N × J matrix) | genotype matrices, linear algebra, vectorised maths |
| pandas | `DataFrame` (labelled table) | expression tables, `groupby`, ranking genes |
| Matplotlib | `Figure`, `Axes` | histograms, scatter plots, heatmaps, line plots |

**Vectorisation in one equation.** The column means of a matrix,

$$\bar x_j = \frac{1}{N} \sum_{i=1}^{N} X_{ij} \quad (j = 1, \dots, J),$$

are a double loop in plain Python but one call in NumPy, `X.mean(axis=0)` — tens to hundreds of times faster,
depending on the computer (Section 3.6 measures it on yours). *In words:* average down every column at once.

> 🎓 **Exam tip.** Programming homework is graded on correct numbers *and* readable code. Name variables after what they
    contain (`allele_freq`, not `a`), write docstrings, and label every plot axis with units.

---
## 7 · Reading error messages (and the bugs that give none)

> 📍 **Where we are.** You have now written real code — and you will make mistakes. Everyone does. The skill is to read
> what Python tells you.

When code fails, Python prints a **traceback**: the chain of calls that led to the problem, ending with the **error
type and message** on the last line. **Read the last line first**, then look up for the line marked with an arrow
(`---->`) — that is where it happened. Below we trigger the five errors you will meet most often. Each is wrapped in
`try:` / `except ... as err:` so that the notebook keeps running and just prints the message.

In [ ]:
# ── 7.1 Five common errors, caught and printed ──
try:
    print(gc_contnet(dna))                       # typo in the function name
except NameError as err:
    print("NameError: ", err)

try:
    print(dna[100])                              # the string has only 27 characters (indices 0–26)
except IndexError as err:
    print("IndexError:", err)

try:
    print(genetic_code["AUG"])                   # RNA letter U, but our dictionary uses DNA letters
except KeyError as err:
    print("KeyError:  ", err, "← this key is not in the dictionary")

try:
    print("GC content: " + gc_content(dna))      # gluing text to a number
except TypeError as err:
    print("TypeError: ", err)

try:
    print(G + np.ones(3))                        # (200, 1000) + (3,): shapes cannot be broadcast
except ValueError as err:
    print("ValueError:", err)

🔍 **What just happened?** Each message names the problem precisely:

* **NameError** — Python does not know the name: a typo (`gc_contnet`), or a cell that defines it has not been run.
* **IndexError** — you asked for a position that does not exist (remember: indices run from 0 to `len − 1`).
* **KeyError** — the dictionary has no such key. Here the codon was written in RNA letters (U instead of T): a
  biologically meaningful bug.
* **TypeError** — an operation got the wrong kind of value: `+` cannot join a string and a number. Fix with an
  f-string, `f"GC content: {gc_content(dna)}"`.
* **ValueError: operands could not be broadcast together with shapes (200,1000) (3,)** — the broadcasting rule of
  Section 3.5 failed: the last dimensions (1000 and 3) neither match nor equal 1. Shape errors are the most common bug in
  numerical code; print `.shape` of everything involved.

### 7.2 ⚠️ The dangerous bugs raise no error

Some mistakes produce **wrong numbers silently**. A classic: our genotype matrix uses `int8`, which can only hold
numbers from −128 to 127. What happens when we add beyond that?

In [ ]:
# ── 7.2 Silent integer overflow (NumPy would normally print a RuntimeWarning; we silence it to see the value) ──
j_common = int(np.argmax(G.sum(axis=0)))           # the SNP whose allele is most common in our 200 people
with np.errstate(over="ignore"):
    print("np.int8(100) + np.int8(100) =", np.int8(100) + np.int8(100), "  (not 200!)")
    wrong_total = np.int8(0)
    for value in G[:, j_common]:
        wrong_total = wrong_total + value          # stays int8 and wraps around past 127
print(f"total count of the allele at SNP {j_common}, int8 running total:", wrong_total)
print("correct total (NumPy's sum uses a large integer type):  ", G[:, j_common].sum())

🔍 **What just happened?** 100 + 100 gave **−56**: the result "wrapped around" because `int8` has no room for 200.
Summing the most common allele's column with an `int8` running total ended at **−44**, although 212 copies are really
present — a negative allele count! NumPy's own `.sum()` automatically switches to a large integer type (64-bit on
most computers) and is correct. (That is why the loop in Section 3.6 converted each value with
`int(...)`.) No red error appeared — only the numbers were wrong.

**Debugging habits that catch both kinds of bug:**

1. **Print shapes and dtypes** (`x.shape`, `x.dtype`) at every step of a new analysis.
2. **Test on a tiny example you can check by hand** (like `revcomp("ATGC")`), and keep the test as an
   `assert`.
3. **Compare with a second method** (our loop vs NumPy, our count vs `str.count`).
4. **Look at the data** — a histogram or `describe()` reveals impossible values (negative counts, frequencies > 1).
5. **Restart and run all** before trusting a result: it guarantees the notebook does not depend on cells run out of
   order or on variables you deleted.
6. **Read the message, then search it.** Copying the last line of a traceback into a search engine usually finds the
   answer.

---
## 8 · Mini-project: Hardy–Weinberg across 100 SNPs

> 📍 **Where we are.** Time to combine everything: simulation (Section 4), arrays, masks and `axis` (Section 3), and
> plotting (Section 6) in one small population-genetics analysis.

**The question.** In a large, randomly mating population, genotype frequencies follow the Hardy–Weinberg proportions
$(1-p)^2$, $2p(1-p)$, $p^2$ (Section 4.2). Geneticists routinely check this for every SNP of a study: big deviations can
flag genotyping errors — or real biology. We will

1. simulate 500 people × 100 SNPs;
2. estimate each SNP's allele frequency $\hat p_j$;
3. compare the **observed** genotype counts with the **expected** counts $N(1-\hat p_j)^2$, $2N\hat p_j(1-\hat p_j)$,
   $N\hat p_j^2$;
4. plot observed versus expected **heterozygosity** (the fraction of heterozygotes).

$$H_{\text{exp},j} = 2\,\hat p_j\,(1-\hat p_j), \qquad H_{\text{obs},j} = \frac{\#\{i : G_{ij} = 1\}}{N}$$

*In words:* the expected heterozygosity of SNP *j* is what Hardy–Weinberg predicts from its allele frequency; the
observed heterozygosity is the fraction of people who actually are heterozygous ("#{…}" means "the number of
individuals *i* for which…").

In [ ]:
# ── 8.1 Observed vs expected genotype counts in one population ──
rng = np.random.default_rng(SEED + 8)
N, J = 500, 100
G1, _, _ = simulate_genotypes(rng, N, J)
p1 = G1.mean(axis=0) / 2                                               # estimated allele frequencies
observed = np.stack([(G1 == g).sum(axis=0) for g in (0, 1, 2)])        # shape (3, J): counts of 0, 1, 2
expected = np.stack([N * (1 - p1) ** 2, N * 2 * p1 * (1 - p1), N * p1 ** 2])
summary = pd.DataFrame({"p_hat": p1[:5].round(3), "obs 0/1/2": [tuple(observed[:, j]) for j in range(5)],
                        "exp 0/1/2": [tuple(expected[:, j].round(1)) for j in range(5)]}, index=[f"SNP {j}" for j in range(5)])
print(summary)
H_obs, H_exp = observed[1] / N, expected[1] / N
print(f"\nmean observed heterozygosity {H_obs.mean():.4f} vs expected {H_exp.mean():.4f}; "
      f"largest |difference| {np.abs(H_obs - H_exp).max():.3f}; SNPs below expectation: {np.mean(H_obs < H_exp):.0%}")

🔍 **What just happened?** For every SNP the observed counts of 0, 1 and 2 are close to the Hardy–Weinberg
expectation (first five SNPs shown). Averaged over the 100 SNPs, observed and expected heterozygosity agree to the
third decimal; individual SNPs scatter by a few percentage points purely by chance, about half above and half below
expectation. Of course: we simulated under Hardy–Weinberg. Now the interesting part.

### 8.2 A hidden population structure

Suppose our 500 people secretly come from **two populations** whose allele frequencies differ (the toolkit draws them
with a genetic differentiation $F_{ST} = 0.1$, a typical value between continental populations). Each population is in
Hardy–Weinberg equilibrium on its own — but we pool them without knowing.

In [ ]:
# ── 8.2 Pool two populations and repeat the check ──
G2, pop2, freqs2 = simulate_genotypes(rng, N, J, n_pops=2, fst=0.1)
p2 = G2.mean(axis=0) / 2
H_obs2, H_exp2 = (G2 == 1).mean(axis=0), 2 * p2 * (1 - p2)
print(f"pooled sample: mean observed {H_obs2.mean():.4f} vs expected {H_exp2.mean():.4f}  "
      f"(deficit {1 - H_obs2.mean() / H_exp2.mean():.1%}); SNPs below expectation: {np.mean(H_obs2 < H_exp2):.0%}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, ho, he, title in [(axes[0], H_obs, H_exp, "one population (SIMULATED)"),
                          (axes[1], H_obs2, H_exp2, "two populations pooled, F_ST = 0.1 (SIMULATED)")]:
    ax.scatter(he, ho, s=18, color="tab:blue", edgecolor="k", lw=0.3)
    ax.plot([0, 0.55], [0, 0.55], "k--", lw=1, label="observed = expected")
    ax.set_xlabel("expected heterozygosity 2p(1−p)"); ax.set_ylabel("observed heterozygosity")
    ax.set_title(title); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the single population (left) the points scatter evenly around the diagonal. In the pooled
sample (right) most points fall **below** it: there are **fewer heterozygotes than Hardy–Weinberg predicts** — about
7 % fewer on average, at roughly three quarters of the SNPs. This is the **Wahlund effect**: when subpopulations with
different allele frequencies are mixed, each subpopulation contributes its own homozygotes, and the pooled allele
frequency overpredicts heterozygotes. Nothing was wrong with any individual genotype; the deficit is a fingerprint of
**hidden population structure**.

That fingerprint matters. In a genome-wide association study, hidden ancestry can create false associations between
SNPs and traits (Module 02 simulates such a confounder), and **principal component analysis** — the method Novembre
*et al.* used to draw the genetic map of Europe — is the standard way to detect and correct it (Module 14).

✏️ **Your turn.** (1) Change `fst=0.1` to `fst=0.01` and then to `fst=0.3`: how does the deficit change? Is it roughly
proportional to $F_{ST}$? (2) Use the population labels `pop2` to compute the heterozygosity *within* population 0 only
(`G2[pop2 == 0]`): does the deficit disappear? (3) Increase N to 5,000 in cell 8.1: what happens to the scatter of the
points around the diagonal, and why?

## 📌 Key takeaways

- Python values: numbers (`int`, `float`), strings (DNA!), booleans; **indices start at 0** and slices exclude their end.
- **Lists** hold ordered items, **dictionaries** map keys to values (the genetic code), **`for` loops** repeat,
  **`if`** decides, **functions** with docstrings package computations — test them with `assert`.
- A **NumPy array** has a **shape** and a **dtype**; the course's data matrix is **N × J** (rows = samples). `axis=0`
  reduces over samples (one value per feature), **masks** select by a condition, **broadcasting** applies a
  per-column operation to every row.
- **Vectorised** NumPy code is far faster and closer to the mathematics than Python loops.
- **Seeded generators** make simulations reproducible; never tune a seed to get the result you want.
- **pandas** DataFrames add labels: select, `describe`, `groupby` and rank — the leukaemia marker genes in a few lines.
  Group means can hide subgroups (the B-cell genes); with simulated data the truth lets you check.
- Every plot needs labelled axes with units and a title that says what to see. **Read the last line of a traceback
  first**; beware silent bugs such as integer overflow.
- Hardy–Weinberg proportions can be checked SNP by SNP; a **heterozygote deficit** across many SNPs reveals hidden
  population structure.

---
## ✅ Checkpoint

1. What do `"GATTACA"[1:4]` and `"GATTACA"[::-1]` return?
   <details><summary>Answer</summary>`"ATT"` (positions 1, 2, 3 — the end index 4 is excluded) and `"ACATTAG"` (the string reversed).</details>
2. `X` has shape (72, 3571). What are the shapes of `X.mean(axis=0)`, `X.mean(axis=1)` and `X[:, :10]`?
   <details><summary>Answer</summary>(3571,) — one mean per gene; (72,) — one mean per patient; (72, 10) — all patients, first ten genes.</details>
3. Why does `X - X.mean(axis=0)` work for a (72, 3571) array, but `X - X.mean(axis=1)` fail? How do you fix the second?
   <details><summary>Answer</summary>Broadcasting compares shapes from the right: (72, 3571) and (3571,) match; (72, 3571) and (72,) do not (3571 ≠ 72). Use `X.mean(axis=1, keepdims=True)`, shape (72, 1), to subtract one value per row.</details>
4. A SNP has allele frequency p = 0.1. Out of 1,000 people, how many heterozygotes and how many homozygotes for the counted allele does Hardy–Weinberg predict?
   <details><summary>Answer</summary>Heterozygotes: 1000 × 2 × 0.1 × 0.9 = 180. Homozygotes (2 copies): 1000 × 0.1² = 10. (And 1000 × 0.9² = 810 with 0 copies; 180 + 10 + 810 = 1000.)</details>
5. What does this print, and why?  `codon = "ATG"; print(codon in ["TAA", "TAG", "TGA"])`
   <details><summary>Answer</summary>`False` — ATG (start codon, methionine) is not in the list of stop codons.</details>
6. Your code prints `KeyError: 'AUG'` when translating. What is the most likely cause?
   <details><summary>Answer</summary>The sequence is RNA (contains U) but the dictionary uses DNA letters (T). Convert first with `seq.replace("U", "T")`.</details>
7. Two notebooks give different simulation results although the code is identical. Name two possible reasons.
   <details><summary>Answer</summary>Different seeds (or no seed); cells run in a different order or a different number of times, so the generator was at a different point in its sequence (or different library versions).</details>
8. Pooling two populations produced fewer heterozygotes than expected. Does that mean genotyping errors? What else could it mean?
   <details><summary>Answer</summary>Not necessarily: it is the Wahlund effect — hidden population structure (different allele frequencies in the subpopulations) creates a heterozygote deficit even when every genotype is correct. Inbreeding has a similar effect.</details>

## 🧠 Questions & discussion

1. Our `translate` function stops at the first stop codon and assumes the reading frame starts at position 1. How would you find *all* open reading frames in a long DNA sequence, on both strands?
2. Vectorised code is faster — but is it always clearer? When might an explicit loop be the better choice, for a reader or for memory?
3. pandas aligns Series by label. Describe a situation in a gene-expression analysis where label alignment protects you from an error, and one where it could surprise you.
4. A genotyping study reports SNPs that 'fail Hardy–Weinberg' and removes them. What could be lost if the population is structured, or if a SNP is truly associated with a disease in a case-only sample?
5. Why is it important that a simulation's random seed is *reported* in a paper, and why is reporting one seed not enough to make a claim?

## 🚀 Bonus — going further

- **All six reading frames.** Write `orfs(seq)` that translates the three frames of `seq` and of `revcomp(seq)` and reports every ORF longer than 50 codons. Test it on `insulin_orf` embedded in random DNA (`''.join(rng.choice(list('ACGT'), 500))`).
- **One-hot DNA.** The toolkit function `one_hot_dna` in [`../course_toolkit.py`](../course_toolkit.py) (used in Module 22 for neural networks on DNA) turns a sequence into an L × 4 array. Re-implement it with a dictionary and a loop, then with NumPy indexing, and compare with the toolkit's version.
- **A quick Hardy–Weinberg test.** For each SNP compute the chi-square statistic Σ (observed − expected)² / expected over the three genotypes. How large does it get in the single population, and in the pooled one? (Module 02 explains what such a statistic means.)
- **More simulated data.** Copy `simulate_signaling_cytometry` (and the names it uses) from [`../course_toolkit.py`](../course_toolkit.py) and generate the single-cell protein-signalling data shaped like the T-cell study of Sachs *et al.* (2005), used in Modules 15–18; explore it with `describe`, histograms and scatter plots.
- **Related notebooks in this repository.** The cryo-ET sister course's Python module: [`../../../../cryoem/tutorial/BiologyToResearch/`](../../../../cryoem/tutorial/BiologyToResearch/) (Module 01 there treats images and 3-D volumes as NumPy arrays).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 01 — Python & NumPy — date:
Input: an insulin ORF (333 nt); simulated genotype matrices (200 × 1,000; 500 × 100);
       SIMULATED leukaemia expression table shaped like Golub et al. 1999 (72 × 3,571)
Output: protein translation and mutation classes; allele frequencies; top ALL/AML genes; observed vs expected heterozygosity plots
Method: strings/dicts/loops/functions; NumPy masks, axis reductions, broadcasting; pandas groupby; Matplotlib
Key equation: H_exp = 2p(1 − p);  p_hat = mean genotype / 2
Key parameters: N, J, SEED, F_ST of the pooled populations
What changed when I modified F_ST / N / the seed:
Ground-truth check: which top genes have a true ALL-vs-AML effect, which only a subtype effect?
One failure mode: (e.g., int8 overflow; broadcasting shape mismatch; RNA letters in a DNA dictionary)
Connection to the 02-620 topics (which idea of the lecture this illustrates):
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the programming prerequisite the lectures assume and the data-matrix conventions of its introductory and regression lectures.
- Harris C. R., Millman K. J., van der Walt S. J. *et al.* (2020) Array programming with NumPy. *Nature* 585:357–362.
- McKinney W. (2010) Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61.
- Hunter J. D. (2007) Matplotlib: a 2D graphics environment. *Computing in Science & Engineering* 9:90–95.
- Golub T. R. *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537.
- The UniProt Consortium. UniProtKB entry P01308 (INS_HUMAN, insulin). https://www.uniprot.org/uniprotkb/P01308
- Hartl D. L., Clark A. G. (2007) *Principles of Population Genetics*, 4th ed. Sinauer — Hardy–Weinberg equilibrium and the Wahlund effect.
- The NumPy documentation (https://numpy.org/doc/) and the pandas documentation (https://pandas.pydata.org/docs/) — user guides and reference pages for every function used in this module.

## ➡️ Next

[Module 02 · Probability: the language of uncertainty](02_probability_for_biologists.ipynb) — you have simulated
    coin flips and genotypes; now you will learn the rules that govern them: distributions, conditional probability,
    Bayes' rule and independence, the foundations of every model in 02-620.